### Research Hypothesis


The paper proposes that combining multiple financial indicators may improve cryptocurrency trading performance.

Rather than evaluating indicators individually, this study tests whether a composite score combining multiple feature families can generate superior risk-adjusted returns.

The following feature families were selected:

- BTC Relative Strength
- Volume
- Volatility
- Bollinger Width

The resulting strategy is compared against S12, the strongest internally developed BTC Relative Strength strategy.

### Feature Selection

Following correlation and IC analysis, representative features were selected from each feature family.

| Feature Family | Selected Feature |
|---------------|------------------|
| Relative Strength | BTC RS |
| Volume | Volume Ratio 20D |
| Volatility | Volatility 20D |
| Volatility Expansion | Bollinger Width |

These features were chosen because they exhibited the greatest independence while avoiding excessive multicollinearity.

### Strategy

In [ ]:
# ============================================================
# R02-S01 Financial Feature Composite
# ============================================================

def strategy(data):

    # --------------------------------------------------------
    # Data
    # --------------------------------------------------------

    close = data.sel(field="close")
    vol = data.sel(field="vol")
    is_liquid = data.sel(field="is_liquid")

    close = xr.where(
        is_liquid == 1,
        close,
        np.nan
    )

    vol = xr.where(
        is_liquid == 1,
        vol,
        np.nan
    )

    btc = data.sel(
        field="close",
        asset="BTC"
    )

    # --------------------------------------------------------
    # BTC Regime
    # --------------------------------------------------------

    btc_sma200 = qnta.sma(
        btc,
        200
    )

    bull_market = xr.where(
        btc > btc_sma200,
        1,
        0
    )

    # --------------------------------------------------------
    # Returns
    # --------------------------------------------------------

    ret = close / close.shift(time=1) - 1

    # --------------------------------------------------------
    # BTC Relative Strength
    # --------------------------------------------------------

    ret_21d = (
        close /
        close.shift(time=21)
        - 1
    )

    btc_ret_21d = (
        btc /
        btc.shift(time=21)
        - 1
    )

    btc_rs = (
        ret_21d
        - btc_ret_21d
    )

    btc_rs_rank = btc_rs.rank("asset")

    # --------------------------------------------------------
    # Volume Feature
    # --------------------------------------------------------

    vol_ma20 = qnta.sma(
        vol,
        20
    )

    volume_ratio = (
        vol /
        (vol_ma20 + 1e-6)
    )

    volume_rank = (
        volume_ratio
        .clip(min=0, max=4)
        .rank("asset")
    )

    # --------------------------------------------------------
    # Volatility Feature
    # --------------------------------------------------------

    vol20 = qnta.std(
        ret,
        20
    )

    vol_rank = vol20.rank(
        "asset"
    )

    # --------------------------------------------------------
    # Bollinger Width
    # --------------------------------------------------------

    bb_mid = qnta.sma(
        close,
        20
    )

    bb_std = qnta.std(
        close,
        20
    )

    bb_width = (
        4 * bb_std
    ) / (
        bb_mid + 1e-6
    )

    bb_rank = bb_width.rank(
        "asset"
    )

    # --------------------------------------------------------
    # Composite Score
    # --------------------------------------------------------

    score = (
        0.40 * btc_rs_rank
        +
        0.30 * volume_rank
        -
        0.15 * vol_rank
        -
        0.15 * bb_rank
    )

    # --------------------------------------------------------
    # Top Assets
    # --------------------------------------------------------

    ranks = score.rank(
        "asset"
    )

    top_assets = xr.where(
        ranks >= 6,
        1,
        0
    )

    weights = score * top_assets

    # --------------------------------------------------------
    # Normalize
    # --------------------------------------------------------

    weights = weights / (
        abs(weights).sum("asset")
        + 1e-6
    )

    # --------------------------------------------------------
    # BTC Regime Filter
    # --------------------------------------------------------

    weights = (
        weights
        * bull_market
    )

    return weights.fillna(0)

### Backtest Execution and submission

In [ ]:
weights = strategy(data)

weights = qnout.clean(
    weights,
    data,
    "crypto_daily_long"
)

qnout.write(weights)

stats = qnstats.calc_stat(
    data,
    weights.sel(time=slice("2016-01-01", None))
)

stats_pd = stats.to_pandas()

### Performance Evaluation

In [ ]:
print(
    stats_pd.iloc[-1][[
        "equity",
        "sharpe_ratio",
        "max_drawdown",
        "avg_turnover",
        "avg_holding_time"
    ]]
)

Final Metrics:
field
equity              5831.083329
sharpe_ratio           1.477464
max_drawdown          -0.853326
avg_turnover           0.328684
avg_holding_time       3.814537
Name: 2026-06-10 00:00:00, dtype: float64

## Comparison with Benchmark-


| Strategy | Sharpe | Max Drawdown |
|----------|---------:|---------:|
| S12 Correlation-Aware BTC RS | 1.715 | -0.693 |
| Paper Feature Composite | 1.477 | -0.853 |

The paper-inspired feature framework underperformed the benchmark strategy on both return and risk-adjusted performance measures.

## Conclusion

The paper-inspired feature framework failed to outperform the internally developed BTC Relative Strength strategy.

Key findings:

- BTC Relative Strength remained the strongest positive factor.
- Volume provided a small positive contribution.
- Volatility and Bollinger Width acted primarily as risk indicators.
- Traditional technical indicators failed to generate superior performance.

The results suggest that crypto-native relative strength signals provide more useful information than traditional technical indicators within the Quantiacs cryptocurrency universe.

Future research may investigate machine-learning implementations of these features, but the factor combination alone does not justify replacing the existing S12 strategy.